# ULEEN — Sweep de Anomalia em IoT (TON_IoT)

Sweep do modelo **ULEEN** (Ultra Low-Energy Edge Networks, Susskind et al. ACM TACO 2023)
sobre as 7 bases de telemetria do **TON_IoT** (UNSW Canberra Cyber).

Diferença-chave em relação ao WiSARD/BloomWiSARD: o ULEEN usa **backprop** para treinar
filtros de Bloom contínuos via STE (Straight-Through Estimator), com um ensemble de
submodelos independentes.

| Eixo | Valores (full) |
|---|---|
| Tipo de termômetro | `linear`, `gaussian` (internos ao ULEEN) |
| Bits por feature (`bits_per_input`) | {4, 8, 16, 32} |
| Bits por filtro / addressSize (`filter_inputs`) | {4, 8, 16, 32} |
| Tamanho do ensemble (`n_submodels`) | {1, 3, 5} |
| Épocas (`epochs`) | {10, 30} |

Fixos: `filter_entries=64`, `filter_hash_functions=2`, `lr=0.01`, `dropout=0.0`.

Resultados em `results/uleen/<base>__<task>.jsonl`.

## 0.1 Setup

In [1]:
import os, json, time, pickle, math, warnings, shutil
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, balanced_accuracy_score,
                             roc_auc_score, confusion_matrix)
from tqdm.auto import tqdm
import torch
import wisardpkg as wp
from wisardpkg.models.uleen import ULEENClassifier

warnings.filterwarnings('ignore')

DATA_DIR    = Path('../../data/toniot/')
RESULTS_DIR = Path('./results/')

EXPECTED = ['Fridge', 'Garage_Door', 'GPS_Tracker', 'Modbus',
            'Motion_Light', 'Thermostat', 'Weather']

# True → roda apenas a primeira combinação válida de hiperparâmetros por base/tarefa
DRY_RUN = False

print(f'wisardpkg {wp.__version__}  |  torch {torch.__version__}')

/home/renan/Git/masters/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


wisardpkg 2.0.0a7  |  torch 2.10.0+cu128


## 1. Carregar e limpar os 7 CSVs

Mesma `clean_df` dos outros notebooks — obrigatória para corrigir espaços
e inconsistências dos CSVs crus do TON_IoT.

In [2]:
def clean_df(df):
    df = df.copy()
    df.columns = [c.strip().lower() for c in df.columns]
    for c in df.columns:
        if not pd.api.types.is_numeric_dtype(df[c]):
            df[c] = df[c].astype(str).str.strip().str.lower()
    if 'sphone_signal' in df.columns:
        df['sphone_signal'] = (df['sphone_signal']
                               .map({'0': 0, '1': 1, 'false': 0, 'true': 1})
                               .astype('Int64'))
    if 'label' in df.columns:
        df['label'] = pd.to_numeric(df['label'], errors='coerce').astype('Int64')
    return df


def find_csv(device):
    if not DATA_DIR.exists():
        return None
    target = f'train_test_iot_{device}.csv'.lower()
    for p in DATA_DIR.iterdir():
        if p.name.lower() == target:
            return p
    return None


dfs = {}
missing = []
for device in EXPECTED:
    p = find_csv(device)
    if p is None:
        missing.append(device)
        continue
    dfs[device] = clean_df(pd.read_csv(p))

print(f'Carregadas {len(dfs)} / {len(EXPECTED)} bases.')
for device, df in dfs.items():
    print(f'  {device:<13}  {df.shape[0]:>6,} x {df.shape[1]}')
if missing:
    print(f'\nFALTANDO: {missing}')
    print(f'Esperado em: {DATA_DIR.resolve()}')
DATA_AVAILABLE = len(dfs) == len(EXPECTED)
assert DATA_AVAILABLE, 'Faltam CSVs — ver README.md'

Carregadas 7 / 7 bases.
  Fridge         39,944 x 6
  Garage_Door    39,587 x 6
  GPS_Tracker    38,960 x 6
  Modbus         31,106 x 8
  Motion_Light   39,488 x 6
  Thermostat     32,774 x 6
  Weather        39,260 x 7


## 2. Identificação de features

In [3]:
META_COLS = {'date', 'time', 'label', 'type'}


def feature_cols(df):
    num, cat = [], []
    for c in df.columns:
        if c in META_COLS:
            continue
        if pd.api.types.is_numeric_dtype(df[c]):
            num.append(c)
        else:
            cat.append(c)
    return num, cat


for device, df in dfs.items():
    num, cat = feature_cols(df)
    # n_features para o ULEEN: num cols + 1 col por cat binária (ou k cols por k-ária)
    n_cat_cols = sum(1 if df[c].nunique() == 2 else df[c].nunique() for c in cat)
    print(f'{device:<13}  num={len(num)}  cat={len(cat)}  n_features_uleen={len(num)+n_cat_cols}')

Fridge         num=1  cat=1  n_features_uleen=2
Garage_Door    num=1  cat=1  n_features_uleen=2
GPS_Tracker    num=2  cat=0  n_features_uleen=2
Modbus         num=4  cat=0  n_features_uleen=4
Motion_Light   num=1  cat=1  n_features_uleen=2
Thermostat     num=2  cat=0  n_features_uleen=2
Weather        num=3  cat=0  n_features_uleen=3


## 3. Preparação de dados para ULEEN

O ULEEN aplica seu próprio termômetro internamente (linear ou gaussiano),
então passamos features **como float**, não como bits pré-codificados.

- **Numéricas**: passadas diretamente como float.
- **Categóricas binárias**: encodadas como 0/1 float — 1 coluna por feature.
  O ULEEN aplicará o termômetro a essa coluna também.

Diferença do WiSARD/BloomWiSARD: aqui `cat_bits = n_cat × bits_per_input`
(não 1 bit por feature binária), porque o ULEEN termometriza todas as colunas.

In [4]:
def cat_categories(df_train, cat_cols):
    return {c: sorted(df_train[c].unique()) for c in cat_cols}


def prepare_X_uleen(df, num_cols, cat_cols, cat_values):
    """Retorna array float (n, n_features) para entrada no ULEENClassifier."""
    parts = []
    if num_cols:
        parts.append(df[num_cols].astype(float).values)
    for c in cat_cols:
        cats = cat_values[c]
        col  = df[c].values
        if len(cats) == 2:
            parts.append((col == cats[1]).astype(float).reshape(-1, 1))
        else:
            for cat in cats:
                parts.append((col == cat).astype(float).reshape(-1, 1))
    return np.hstack(parts) if parts else np.zeros((len(df), 0), dtype=float)

## 4. Split estratificado

In [5]:
def make_split(df, task, test_size=0.3, random_state=0):
    if task == 'binary':
        y = df['label'].astype(int).map({0: 'normal', 1: 'attack'}).values
    elif task == 'multiclass':
        y = df['type'].values
    else:
        raise ValueError(task)
    idx_train, idx_test = train_test_split(
        np.arange(len(df)),
        test_size=test_size,
        random_state=random_state,
        stratify=df['type'].values,
    )
    return (df.iloc[idx_train].reset_index(drop=True),
            df.iloc[idx_test].reset_index(drop=True),
            y[idx_train], y[idx_test])

## 5. Métricas (schema PLANO §4)

In [6]:
def compute_metrics(y_true, y_pred, y_score=None, task='binary',
                    score_classes=None):
    out = {
        'accuracy':           float(accuracy_score(y_true, y_pred)),
        'precision_macro':    float(precision_score(y_true, y_pred, average='macro',    zero_division=0)),
        'precision_weighted': float(precision_score(y_true, y_pred, average='weighted', zero_division=0)),
        'recall_macro':       float(recall_score(y_true, y_pred, average='macro',    zero_division=0)),
        'recall_weighted':    float(recall_score(y_true, y_pred, average='weighted', zero_division=0)),
        'f1_macro':           float(f1_score(y_true, y_pred, average='macro',    zero_division=0)),
        'f1_weighted':        float(f1_score(y_true, y_pred, average='weighted', zero_division=0)),
        'balanced_accuracy':  float(balanced_accuracy_score(y_true, y_pred)),
        'auc_roc':            None,
    }
    if y_score is not None:
        try:
            score_arr = np.asarray(y_score, dtype=float)
            if task == 'binary':
                pos_idx   = score_classes.index('attack')
                pos_score = score_arr[:, pos_idx]
                out['auc_roc'] = float(roc_auc_score(y_true == 'attack', pos_score))
            else:
                out['auc_roc'] = float(roc_auc_score(
                    y_true, score_arr, multi_class='ovr', average='macro',
                    labels=score_classes,
                ))
        except Exception:
            pass
    labels = sorted(set(list(y_true) + list(y_pred)))
    cm = confusion_matrix(y_true, y_pred, labels=labels).tolist()
    return out, cm, labels


def uleen_scores(clf, X_uleen):
    """Logits por classe do ensemble — usados para AUC-ROC."""
    Xb = clf._binarise(np.asarray(X_uleen, dtype=np.float32))
    clf.model.eval()
    with torch.no_grad():
        results = clf.model(Xb)          # (n_submodels, batch, n_classes)
        logits  = results.sum(axis=0).cpu().numpy()   # (batch, n_classes)
    return logits

## 6. Memória e latência — ULEEN

**Memória serializada**: `pickle` do modelo PyTorch.
**Memória teórica**: tabela Bloom binarizada pós-treino, conforme paper Table 2
(`deployed_size_bytes()` sem metadados).

In [7]:
def measure_memory_uleen(clf):
    try:
        serialized = len(pickle.dumps(clf.model))
    except Exception:
        serialized = clf.deployed_size_bytes()
    theoretical = clf.deployed_size_bytes()
    return serialized, theoretical


def measure_latency_uleen(clf, X_test_rows, n_iter=200):
    """Mediana de predict() single-sample em microssegundos."""
    if len(X_test_rows) == 0:
        return None
    n = len(X_test_rows)
    pool = [X_test_rows[i:i+1] for i in range(min(n, n_iter))]
    for x in pool[:min(20, len(pool))]:
        _ = clf.predict(x)
    times = []
    for i in range(n_iter):
        x = pool[i % len(pool)]
        t0 = time.perf_counter_ns()
        _ = clf.predict(x)
        times.append((time.perf_counter_ns() - t0) / 1000.0)
    times.sort()
    return float(times[len(times) // 2])

## 7. Grid de hiperparâmetros — ULEEN

Grid menor que os outros modelos (PLANO §3.2): treinamento com backprop é
mais caro. Foco em `n_submodels` e `epochs`; eixos comuns podados de 6/8 → 4.

`filter_inputs` = addressSize (bits por filtro Bloom = tamanho da tupla).
`bits_per_input` = thermoSize (bits por feature, codificação interna do ULEEN).

In [8]:
# ─── Hiperparâmetros fixos ───────────────────────────────────────────────────
FILTER_ENTRIES      = 64   # tamanho da tabela Bloom (potência de 2)
FILTER_HASH_FUNCS   = 2    # funções hash H3 por filtro
LEARNING_RATE       = 0.01
DROPOUT             = 0.0
BATCH_SIZE          = 32
DECAY_LR            = True

# ─── Grid varrido (DRY_RUN vs completo) ─────────────────────────────────────
if DRY_RUN:  # Grid reduzido para teste rápido
    THERMOMETERS   = ['gaussian']
    THERMO_SIZES   = [4, 16]
    ADDRESS_SIZES  = [4, 8]
    ENSEMBLE_SIZES = [1, 3]
    EPOCHS_LIST    = [10]
else:        # Grid completo (PLANO §3.1 + §3.2 ULEEN)
    THERMOMETERS   = ['linear', 'gaussian']
    THERMO_SIZES   = [4, 8, 16]   # podado de 6 → 4 (PLANO §3.2)
    # THERMO_SIZES   = [4, 8, 16, 32]   # podado de 6 → 4 (PLANO §3.2)
    ADDRESS_SIZES  = [4, 8, 16]   # podado de 8 → 4 (PLANO §3.2)
    # ADDRESS_SIZES  = [4, 8, 16, 32]   # podado de 8 → 4 (PLANO §3.2)
    ENSEMBLE_SIZES = [1, 3]
    # ENSEMBLE_SIZES = [1, 3, 5]
    EPOCHS_LIST    = [10, 30]


def iter_grid_uleen(n_features,
                    thermo_sizes=THERMO_SIZES,
                    address_sizes=ADDRESS_SIZES,
                    thermometers=THERMOMETERS,
                    ensemble_sizes=ENSEMBLE_SIZES,
                    epochs_list=EPOCHS_LIST):
    for th in thermometers:
        for ts in thermo_sizes:
            input_bits = n_features * ts
            for ad in address_sizes:
                skipped = ad > input_bits
                for ns in ensemble_sizes:
                    for ep in epochs_list:
                        yield {
                            'thermometer':  th,
                            'thermo_size':  ts,
                            'address_size': ad,
                            'input_bits':   input_bits,
                            'n_submodels':  ns,
                            'epochs':       ep,
                            'skipped':      skipped,
                        }


print('Configs por base (por tarefa):')
for device, df in dfs.items():
    num, cat = feature_cols(df)
    n_cat_cols = sum(1 if df[c].nunique() == 2 else df[c].nunique() for c in cat)
    n_feat = len(num) + n_cat_cols
    total  = sum(1 for _ in iter_grid_uleen(n_feat))
    valid  = sum(1 for c in iter_grid_uleen(n_feat) if not c['skipped'])
    print(f'  {device:<13}  n_feat={n_feat}  total={total:>5,}  validas={valid:>5,}  skip={total-valid:>4,}')

Configs por base (por tarefa):
  Fridge         n_feat=2  total=   72  validas=   64  skip=   8
  Garage_Door    n_feat=2  total=   72  validas=   64  skip=   8
  GPS_Tracker    n_feat=2  total=   72  validas=   64  skip=   8
  Modbus         n_feat=4  total=   72  validas=   72  skip=   0
  Motion_Light   n_feat=2  total=   72  validas=   64  skip=   8
  Thermostat     n_feat=2  total=   72  validas=   64  skip=   8
  Weather        n_feat=3  total=   72  validas=   64  skip=   8


## 8. Schema de resultado (PLANO §4)

In [9]:
def result_dict(model_name, base, task, config, metrics, cm, labels,
                input_info, model_hyperparams=None, machine=None,
                skipped=False, skipped_reason=None):
    return {
        'model':             model_name,
        'base':              base,
        'task':              task,
        'encoder':           {'type': config['thermometer'], 'size': config['thermo_size']},
        'addressSize':       config['address_size'],
        'model_hyperparams': model_hyperparams or {},
        'split':             {'random_state': 0, 'test_size': 0.3, 'stratified': True},
        'input':             input_info,
        'skipped':           skipped,
        'skipped_reason':    skipped_reason,
        'metrics':           metrics,
        'confusion_matrix':  cm,
        'labels':            labels,
        'machine':           machine,
        'wisardpkg_version': wp.__version__,
        'timestamp':         datetime.now(timezone.utc).isoformat(timespec='seconds'),
    }


def save_result(result, model_name, base, task):
    out_dir = RESULTS_DIR / model_name.lower()
    out_dir.mkdir(parents=True, exist_ok=True)
    fname = f'{base}__{task}.jsonl'
    with open(out_dir / fname, 'a') as f:
        f.write(json.dumps(result) + '\n')
    return out_dir / fname

## 9. Sweep completo — ULEEN

Roda todas as bases × tarefas × configs do grid. Configs com
`addressSize > input_bits` são registradas com `skipped=true`.

> **Re-rodar limpa** os JSONLs anteriores para evitar duplicatas.

In [10]:
MACHINE = 'local'   # ajuste: 'colab-cpu', 'local', etc.
TASKS   = ['binary', 'multiclass']

uleen_results_dir = RESULTS_DIR / 'uleen'
if uleen_results_dir.exists():
    shutil.rmtree(uleen_results_dir)
uleen_results_dir.mkdir(parents=True, exist_ok=True)

EMPTY_METRICS = {k: None for k in [
    'accuracy', 'precision_macro', 'precision_weighted',
    'recall_macro', 'recall_weighted', 'f1_macro', 'f1_weighted',
    'balanced_accuracy', 'auc_roc',
    'memory_bytes_serialized', 'memory_bytes_theoretical',
    'train_time_s', 'inference_latency_us',
]}

for device in EXPECTED:
    df = dfs[device]
    num, cat = feature_cols(df)

    for task in TASKS:
        df_train, df_test, y_train, y_test = make_split(df, task)
        cat_values = cat_categories(df_train, cat)

        X_train = prepare_X_uleen(df_train, num, cat, cat_values)
        X_test  = prepare_X_uleen(df_test,  num, cat, cat_values)
        n_feat  = X_train.shape[1]       # cols reais vistas pelo ULEEN
        n_cat_cols = n_feat - len(num)
        classes = sorted(set(y_train))

        grid = list(iter_grid_uleen(n_feat))
        pbar = tqdm(grid, desc=f'{device}/{task}', leave=False)

        for config in pbar:
            input_info = {
                'n_features_num': len(num),
                'n_features_cat': len(cat),
                'cat_bits':  n_cat_cols * config['thermo_size'],  # ULEEN termometriza todas
                'bits_total': config['input_bits'],
            }
            hp = {
                'n_submodels':           config['n_submodels'],
                'epochs':                config['epochs'],
                'filter_entries':        FILTER_ENTRIES,
                'filter_hash_functions': FILTER_HASH_FUNCS,
                'learning_rate':         LEARNING_RATE,
                'dropout_p':             DROPOUT,
            }

            if config['skipped']:
                res = result_dict(
                    'ULEEN', device, task, config,
                    metrics=EMPTY_METRICS, cm=[], labels=[],
                    input_info=input_info, model_hyperparams=hp,
                    machine=MACHINE, skipped=True,
                    skipped_reason='addressSize > input_bits',
                )
                save_result(res, 'ULEEN', device, task)
                continue

            clf = ULEENClassifier(
                bits_per_input        = config['thermo_size'],
                filter_inputs         = config['address_size'],
                filter_entries        = FILTER_ENTRIES,
                filter_hash_functions = FILTER_HASH_FUNCS,
                n_submodels           = config['n_submodels'],
                dropout_p             = DROPOUT,
                epochs                = config['epochs'],
                lr                    = LEARNING_RATE,
                batch_size            = BATCH_SIZE,
                decay_lr              = DECAY_LR,
                thermometer           = config['thermometer'],
            )

            t0 = time.perf_counter()
            clf.fit(X_train, y_train, seed=0)
            train_time_s = time.perf_counter() - t0

            y_pred  = clf.predict(X_test)
            y_score = uleen_scores(clf, X_test)

            metrics, cm, labels = compute_metrics(
                y_test, y_pred, y_score=y_score,
                task=task, score_classes=clf.classes_,
            )

            ser, theo = measure_memory_uleen(clf)
            lat = measure_latency_uleen(clf, X_test[:200], n_iter=200)

            metrics.update({
                'memory_bytes_serialized':  int(ser),
                'memory_bytes_theoretical': int(theo),
                'train_time_s':             round(train_time_s, 4),
                'inference_latency_us':     round(lat, 2) if lat is not None else None,
            })

            res = result_dict(
                'ULEEN', device, task, config,
                metrics=metrics, cm=cm, labels=labels,
                input_info=input_info, model_hyperparams=hp,
                machine=MACHINE,
            )
            save_result(res, 'ULEEN', device, task)

        pbar.close()
        jsonl = uleen_results_dir / f'{device}__{task}.jsonl'
        if jsonl.exists():
            lines = jsonl.read_text().strip().splitlines()
            done  = sum(1 for l in lines if not json.loads(l).get('skipped'))
            skip  = len(lines) - done
            print(f'{device:<13} {task:<11}  {done:>5} rodados  {skip:>5} skipped')

print('\nSweep concluído.')

Fridge        binary          64 rodados      8 skipped


Fridge        multiclass      64 rodados      8 skipped


Garage_Door   binary          64 rodados      8 skipped


Garage_Door   multiclass      64 rodados      8 skipped


GPS_Tracker   binary          64 rodados      8 skipped


...

Motion_Light  multiclass      64 rodados      8 skipped


Thermostat    binary          64 rodados      8 skipped


Thermostat    multiclass      64 rodados      8 skipped


Weather       binary          64 rodados      8 skipped


Weather       multiclass      64 rodados      8 skipped

Sweep concluído.


## 10. Análise rápida dos resultados

Top-5 configs por base e tarefa ordenadas por F1 macro (excluindo configs puladas).
Para análise completa com fronteira de Pareto, ver `03_analise_final.ipynb`.

In [11]:
def load_results(model_name='uleen'):
    rows = []
    res_dir = RESULTS_DIR / model_name
    if not res_dir.exists():
        return pd.DataFrame()
    for fpath in sorted(res_dir.glob('*.jsonl')):
        for line in fpath.read_text().strip().splitlines():
            rows.append(json.loads(line))
    return pd.DataFrame(rows)


res = load_results()
if res.empty:
    print('Nenhum resultado ainda — rode a célula do sweep primeiro.')
else:
    metrics_df = pd.json_normalize(res['metrics'])
    uleen_df = pd.concat([
        res[['model', 'base', 'task', 'addressSize']].reset_index(drop=True),
        pd.json_normalize(res['encoder']).add_prefix('enc_'),
        pd.json_normalize(res['model_hyperparams']).add_prefix('hp_'),
        metrics_df[['accuracy', 'f1_macro', 'balanced_accuracy',
                    'memory_bytes_theoretical', 'train_time_s']],
        res['skipped'].reset_index(drop=True),
    ], axis=1)

    print('=== Top-5 por base x tarefa (F1 macro, sem skips) ===\n')
    for (base, task), grp in uleen_df[~uleen_df['skipped']].groupby(['base', 'task']):
        top = (grp.nlargest(5, 'f1_macro')
               [['enc_type', 'enc_size', 'addressSize',
                 'hp_n_submodels', 'hp_epochs',
                 'accuracy', 'f1_macro', 'balanced_accuracy',
                 'memory_bytes_theoretical', 'train_time_s']]
               .round(4))
        print(f'--- {base} / {task} ---')
        print(top.to_string(index=False))
        print()

=== Top-5 por base x tarefa (F1 macro, sem skips) ===

--- Fridge / binary ---
enc_type  enc_size  addressSize  hp_n_submodels  hp_epochs  accuracy  f1_macro  balanced_accuracy  memory_bytes_theoretical  train_time_s
  linear        16            8               3         10    0.6100    0.4316             0.5016                     192.0       10.5148
  linear        16            8               3         30    0.6170    0.4167             0.5022                     192.0       31.5110
  linear         4            4               1         10    0.6245    0.3844             0.5000                      32.0        6.0287
  linear         4            4               1         30    0.6245    0.3844             0.5000                      32.0       14.1697
  linear         4            4               3         10    0.6245    0.3844             0.5000                      96.0        9.7618

--- Fridge / multiclass ---
enc_type  enc_size  addressSize  hp_n_submodels  hp_epochs  accu